# EfficientNet-B0 with Entropy Filtering, Ordinal Loss + Focal Loss (γ=0.2), and CLAHE-YUV Preprocessing

This notebook trains an EfficientNet-B0 model combining:
- **Entropy filtering**: Removes high-entropy samples for cleaner training
- **Ordinal loss**: Plain BCE on ordinal-encoded ISUP targets — enforces grade ordering
- **Focal loss (γ=0.2)**: Applied on top of the same ordinal predictions — mild focus on hard examples
- **Combined loss**: `total = ordinal_loss + focal_loss`
- **CLAHE-YUV preprocessing**: Contrast Limited Adaptive Histogram Equalization applied exclusively to the Y (luminance) channel of the YUV colour space — enhances cellular structure contrast while preserving H&E stain colours (U/V channels untouched)

In [ ]:
import sys
import os

ROOT_DIR = '../../..'
sys.path.insert(0, os.path.join(ROOT_DIR, 'utils/HEnorm'))

from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import albumentations as Albu
import cv2
import pandas as pd
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler, SequentialSampler
from torch.amp import autocast, GradScaler
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, recall_score, precision_score
from tqdm import tqdm
import optuna
from optuna.pruners import MedianPruner
import sys
sys.path.append("../../..")
from utils.dataset import PandasDataset
from utils.models import EfficientNetApi

## Configuration

In [ ]:
seed = 42
batch_size = 3
num_workers = 4
output_classes = 5
init_lr = 3e-4
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
dropout_rate = 0.6
patience = 7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

data_dir = '../../../..'
images_dir = os.path.join(data_dir, 'tiles')

os.makedirs('../logs', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('models', exist_ok=True)
model_path = 'models/b0-entropy-ordinal-focal-yhuv-clahe.pth'
log_path = 'logs/b0-entropy-ordinal-focal-yhuv-clahe.txt'

# ── Optuna ────────────────────────────────────────────────────────────────────
OPTUNA_SUBSET_FRAC = 0.25   # fração do train usada em cada trial
N_OPTUNA_TRIALS    = 30     # número de trials
N_OPTUNA_EPOCHS    = 5      # épocas por trial
OPTUNA_BATCH_SIZE  = batch_size

STUDY_DB   = 'sqlite:///logs/b0-entropy-ordinal-focal-yhu-optuna.db'
STUDY_NAME = 'b0-entropy-ordinal-focal-yhu'

## CLAHE-YUV Preprocessing

CLAHE (Contrast Limited Adaptive Histogram Equalization) is applied **exclusively to the Y channel** of the YUV colour space.

**Why YUV?**
- **Colour preservation**: In RGB all channels carry colour information; equalising them directly distorts the H&E staining.
- **Luminance focus**: The Y channel encodes light intensity (luminance) — the dimension that reveals cellular structure boundaries. The U and V channels carry chrominance and are left untouched.
- **Processing pipeline**: RGB → YUV → CLAHE on Y → YUV → RGB (input-compatible with the neural network).

In [3]:
from albumentations.core.transforms_interface import ImageOnlyTransform

class CLAHEOnYChannel(ImageOnlyTransform):
    """CLAHE applied to the Y (luminance) channel of YUV colour space.

    Pipeline: RGB → YUV → CLAHE(Y) → YUV → RGB
    Colour information (U/V) is preserved; only contrast in the luminance
    channel is enhanced to highlight cellular structures.
    """

    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8), always_apply=False, p=0.5):
        super().__init__(always_apply=always_apply, p=p)
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size
        self._clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def apply(self, image, **kwargs):
        yuv = cv2.cvtColor(image, cv2.COLOR_RGB2YUV)
        yuv[:, :, 0] = self._clahe.apply(yuv[:, :, 0])
        return cv2.cvtColor(yuv, cv2.COLOR_YUV2RGB)

    def get_transform_init_args_dict(self):
        return {"clip_limit": self.clip_limit, "tile_grid_size": self.tile_grid_size}

## Loss: Ordinal Loss + Focal Loss (γ=0.2)

Two separate terms are combined:

1. **`OrdinalLoss`** — plain BCE applied to ordinal-encoded targets.  Provides the
   base ordinal regression signal that enforces the grade ordering.

2. **`OrdinalFocalLoss`** — focal-weighted BCE on the same ordinal targets with γ=0.2.
   Mildly focuses learning on hard threshold boundaries without discarding easy ones.

```
total_loss = ordinal_loss + focal_loss
           = BCE(sigmoid(logits), ordinal_targets)
           + alpha * (1 - p_t)^0.2 * BCE(sigmoid(logits), ordinal_targets)
```

In [ ]:
class FocalOrdinalRegressionLoss(nn.Module):
    def __init__(self, num_classes=5, alpha=1.0, beta=1.0, gamma=2.0):
        """
        num_classes : número de thresholds ordinais (ISUP: 5)
        alpha       : peso da focal loss
        beta        : peso da ordinal loss
        gamma       : fator de modulação focal
        """
        super().__init__()
        self.alpha       = alpha
        self.beta        = beta
        self.gamma       = gamma
        self.num_classes = num_classes
        self.C2          = num_classes ** 2

    def focal_loss(self, logits, targets):
        probs  = torch.sigmoid(logits)
        bce    = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t    = probs * targets + (1 - probs) * (1 - targets)
        return (((1 - p_t) ** self.gamma) * bce).mean()

    def ordinal_loss(self, logits, targets):
        probs          = torch.sigmoid(logits)
        expected_class = probs.sum(dim=1)
        targets_class  = targets.sum(dim=1)
        squared_error  = (expected_class - targets_class) ** 2
        return squared_error.mean() / self.C2

    def forward(self, logits, targets):
        targets = targets.to(logits.device)
        f_loss  = self.focal_loss(logits, targets)
        o_loss  = self.ordinal_loss(logits, targets)
        return self.alpha * f_loss + self.beta * o_loss


def decode_ordinal_predictions(logits):
    return (torch.sigmoid(logits) > 0.5).sum(dim=1)


print("FocalOrdinalRegressionLoss defined.")

## Load Data with Entropy Filtering

In [5]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
print(f"Original records: {len(df_train_)}")

df_entropy = pd.read_csv(f"{ROOT_DIR}/data/entropy.csv")
print(f"High-entropy samples to remove: {len(df_entropy)}")

df_entropy_sorted = df_entropy.sort_values(by='difficulty_score', ascending=False).reset_index(drop=True)
n_remove          = int(len(df_entropy) * 0.2)
df_entropy_top    = df_entropy_sorted.head(n_remove)

df_train_ = df_train_[~df_train_['image_id'].isin(df_entropy_top['image_id'])].reset_index(drop=True)
print(f"Filtered records: {len(df_train_)}")

df_train_.columns = df_train_.columns.str.strip()

train_indexes = np.where(df_train_['fold'] != 3)[0]
valid_indexes = np.where(df_train_['fold'] == 3)[0]

df_train = df_train_.loc[train_indexes].reset_index(drop=True)
df_val   = df_train_.loc[valid_indexes].reset_index(drop=True)
df_test  = pd.read_csv(f"{ROOT_DIR}/data/test.csv")


def remove_nonexistent_images(df, images_dir):
    paths     = df['image_id'].apply(lambda x: os.path.join(images_dir, f"{x}.png"))
    existing  = [os.path.isfile(p) for p in paths]
    return df[existing]


df_train = remove_nonexistent_images(df_train, images_dir)
df_val   = remove_nonexistent_images(df_val,   images_dir)
df_test  = remove_nonexistent_images(df_test,  images_dir)

print(f"\nTrain:      {len(df_train)} samples")
print(f"Validation: {len(df_val)} samples")
print(f"Test:       {len(df_test)} samples")
print(f"\nTrain class distribution:")
print(df_train['isup_grade'].value_counts().sort_index())

Original records: 9024
High-entropy samples to remove: 903
Filtered records: 8844

Train:      7073 samples
Validation: 1767 samples
Test:       1590 samples

Train class distribution:
isup_grade
0    1953
1    1802
2     899
3     826
4     832
5     761
Name: count, dtype: int64


## Data Augmentation with CLAHE-YUV

`CLAHEOnYChannel` enhances luminance contrast (p=1.0 — applied to every sample) alongside the standard geometric augmentations. The colour channels are unmodified.

In [6]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    CLAHEOnYChannel(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
])

val_transforms = Albu.Compose([
    CLAHEOnYChannel(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0),
])

/tmp/ipykernel_312592/2850931843.py:12: UserWarning: Argument(s) 'always_apply' are not valid for transform BasicTransform
  super().__init__(always_apply=always_apply, p=p)


## Create Datasets and DataLoaders

In [ ]:
train_dataset = PandasDataset(images_dir, df_train, transforms=train_transforms, format="png")
valid_dataset = PandasDataset(images_dir, df_val,   transforms=val_transforms,   format="png")
test_dataset  = PandasDataset(images_dir, df_test,  transforms=val_transforms,   format="png")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    sampler=RandomSampler(train_dataset),
    pin_memory=True, persistent_workers=True)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    sampler=SequentialSampler(valid_dataset),
    pin_memory=True, persistent_workers=True)
test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    sampler=SequentialSampler(test_dataset),
    pin_memory=True, persistent_workers=True)

print(f"Train batches:      {len(train_loader)}")
print(f"Validation batches: {len(valid_loader)}")
print(f"Test batches:       {len(test_loader)}")

## Busca de Hiperparâmetros com Optuna

O Optuna busca simultaneamente:
- **Loss**: `gamma`, `alpha` (w_focal), `beta` (w_ord) da `FocalOrdinalRegressionLoss`
- **Otimizador**: `lr` (log-scale) e `weight_decay`

Treina por `N_OPTUNA_EPOCHS` épocas com `MedianPruner` e maximiza o **QWK** de validação.

In [ ]:
# Subset fixo do train para o Optuna (reprodutível)
rng_optuna = np.random.default_rng(seed)
optuna_idx = rng_optuna.choice(len(df_train), size=int(len(df_train) * OPTUNA_SUBSET_FRAC), replace=False)
df_optuna  = df_train.iloc[optuna_idx].reset_index(drop=True)

optuna_train_ds = PandasDataset(images_dir, df_optuna, transforms=train_transforms, format="png")
optuna_val_ds   = PandasDataset(images_dir, df_val,    transforms=val_transforms,   format="png")

optuna_train_loader = DataLoader(
    optuna_train_ds,
    batch_size=OPTUNA_BATCH_SIZE,
    num_workers=num_workers,
    sampler=RandomSampler(optuna_train_ds),
    pin_memory=True,
)
optuna_val_loader = DataLoader(
    optuna_val_ds,
    batch_size=OPTUNA_BATCH_SIZE,
    num_workers=num_workers,
    sampler=SequentialSampler(optuna_val_ds),
    pin_memory=True,
)

print(f"Optuna subset: {len(df_optuna)} imgs ({OPTUNA_SUBSET_FRAC:.0%} do train)")
print(f"Train batches: {len(optuna_train_loader)} | Val batches: {len(optuna_val_loader)}")

In [ ]:
def build_model(dr=dropout_rate):
    backbone = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model = EfficientNetApi(model=backbone, output_dimensions=output_classes, dropout_rate=dr)
    return model.to(device)


def optuna_train_one_epoch(model, dataloader, optimizer, loss_fn, scaler_opt):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for batch_data, batch_targets, _ in dataloader:
        batch_data    = batch_data.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)
        with autocast(device_type='cuda'):
            logits = model(batch_data)
            loss   = loss_fn(logits, batch_targets)
        scaler_opt.scale(loss).backward()
        scaler_opt.step(optimizer)
        scaler_opt.update()
        optimizer.zero_grad(set_to_none=True)


def optuna_validate(model, dataloader, loss_fn):
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad(), autocast(device_type='cuda'):
        for batch_data, batch_targets, _ in dataloader:
            batch_data    = batch_data.to(device, non_blocking=True)
            batch_targets = batch_targets.to(device, non_blocking=True)
            logits        = model(batch_data)
            preds         = decode_ordinal_predictions(logits)
            targets_cls   = batch_targets.sum(dim=1).long()
            all_preds.append(preds.cpu())
            all_targets.append(targets_cls.cpu())
    all_preds   = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    return cohen_kappa_score(all_targets, all_preds, weights='quadratic')


def objective(trial: optuna.Trial) -> float:
    gamma        = trial.suggest_float('gamma',        0.5, 3.0)
    alpha        = trial.suggest_float('alpha',        0.1, 2.0)
    beta         = trial.suggest_float('beta',         0.1, 2.0)
    lr           = trial.suggest_float('lr',           1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)

    model      = build_model()
    loss_fn    = FocalOrdinalRegressionLoss(alpha=alpha, beta=beta, gamma=gamma)
    opt        = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scaler_opt = GradScaler()

    best_kappa = 0.0
    for epoch in range(N_OPTUNA_EPOCHS):
        optuna_train_one_epoch(model, optuna_train_loader, opt, loss_fn, scaler_opt)
        kappa = optuna_validate(model, optuna_val_loader, loss_fn)

        trial.report(kappa, epoch)
        if trial.should_prune():
            del model
            torch.cuda.empty_cache()
            raise optuna.exceptions.TrialPruned()

        best_kappa = max(best_kappa, kappa)
        print(f"  [Trial {trial.number}, Epoch {epoch+1}] Kappa: {kappa:.4f}")

    del model
    torch.cuda.empty_cache()
    return best_kappa


print("build_model e objective Optuna definidos.")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    storage=STUDY_DB,
    load_if_exists=True,
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)

study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

best = study.best_trial
print(f"\nMelhor trial: #{best.number}")
print(f"  Kappa: {best.value:.4f}")
print(f"  gamma:        {best.params['gamma']:.4f}")
print(f"  alpha:        {best.params['alpha']:.4f}")
print(f"  beta:         {best.params['beta']:.4f}")
print(f"  lr:           {best.params['lr']:.2e}")
print(f"  weight_decay: {best.params['weight_decay']:.2e}")

In [ ]:
importances = optuna.importance.get_param_importances(study)
print("Importância dos hiperparâmetros:")
for param, imp in importances.items():
    print(f"  {param}: {imp:.4f}")

BEST_GAMMA        = best.params['gamma']
BEST_ALPHA        = best.params['alpha']
BEST_BETA         = best.params['beta']
BEST_LR           = best.params['lr']
BEST_WEIGHT_DECAY = best.params['weight_decay']

print(f"\nMelhores hiperparâmetros:")
print(f"  gamma={BEST_GAMMA:.4f}, alpha={BEST_ALPHA:.4f}, beta={BEST_BETA:.4f}")
print(f"  lr={BEST_LR:.2e}, weight_decay={BEST_WEIGHT_DECAY:.2e}")

### Visualizações Optuna

In [ ]:
from optuna import visualization as optvis

fig = optvis.plot_optimization_history(study)
fig.update_layout(title='Histórico de Otimização — Kappa por Trial', height=450)
fig.show()

fig = optvis.plot_param_importances(study)
fig.update_layout(title='Importância dos Hiperparâmetros (fANOVA)', height=400)
fig.show()

fig = optvis.plot_parallel_coordinate(study, params=['gamma', 'alpha', 'beta', 'lr', 'weight_decay'])
fig.update_layout(title='Coordenadas Paralelas — Hiperparâmetros vs Kappa', height=500)
fig.show()

for p1, p2 in [('gamma', 'alpha'), ('gamma', 'beta'), ('alpha', 'beta'), ('lr', 'weight_decay')]:
    fig = optvis.plot_contour(study, params=[p1, p2])
    fig.update_layout(title=f'Contorno: {p1} × {p2}', height=500)
    fig.show()

fig = optvis.plot_slice(study, params=['gamma', 'alpha', 'beta', 'lr', 'weight_decay'])
fig.update_layout(title='Efeito Marginal de Cada Hiperparâmetro', height=450)
fig.show()

## Treino Completo com Melhores Hiperparâmetros do Optuna

In [ ]:
model = build_model(dr=dropout_rate)

loss_function = FocalOrdinalRegressionLoss(
    gamma=BEST_GAMMA,
    alpha=BEST_ALPHA,
    beta=BEST_BETA,
)

print(f"Model loaded on {device}")
print(f"Output dimensions: {output_classes} (ordinal thresholds)")
print(f"Dropout rate: {dropout_rate}")
print(f"Loss: gamma={BEST_GAMMA:.4f}, alpha={BEST_ALPHA:.4f}, beta={BEST_BETA:.4f}")
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Params treináveis: {trainable:,} / {total:,}")

## Optimizer e Scheduler (com hiperparâmetros do Optuna)

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=BEST_LR / warmup_factor, weight_decay=BEST_WEIGHT_DECAY)

scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(
    optimizer,
    multiplier=warmup_factor,
    total_epoch=warmup_epochs,
    after_scheduler=scheduler_cosine
)

scaler = GradScaler()

print(f"Optimizer: Adam | lr={BEST_LR:.2e} | weight_decay={BEST_WEIGHT_DECAY:.2e}")
print("Scheduler: warmup + CosineAnnealingLR configured")

## Training and Validation Functions

In [ ]:
def training_step(model, dataloader, optimizer, device, loss_fn, scaler):
    model.train()
    train_loss = []
    running_loss = 0.0
    bar_progress = tqdm(dataloader, desc="Training")

    for step, (batch_data, batch_targets, _) in enumerate(bar_progress):
        batch_data    = batch_data.to(device, non_blocking=True)
        batch_targets = batch_targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        
        with autocast(device_type='cuda'):
            logits = model(batch_data)
            loss = loss_fn(logits, batch_targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        loss_value = loss.item()
        train_loss.append(loss_value)

        running_loss += loss_value
        smooth_loss = running_loss / (step + 1)

        bar_progress.set_postfix({
            'loss': f'{loss_value:.5f}',
            'smooth': f'{smooth_loss:.5f}'
        })

    return train_loss


def validation_step(model, dataloader, device, loss_fn):
    model.eval()

    validation_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad(), autocast(device_type='cuda'):
        for batch_data, batch_targets, _ in tqdm(dataloader, desc="Validation", leave=False):
            batch_data = batch_data.to(device, non_blocking=True)
            batch_targets = batch_targets.to(device, non_blocking=True)

            logits = model(batch_data)
            loss = loss_fn(logits, batch_targets)

            predictions = decode_ordinal_predictions(logits)
            targets_class = batch_targets.sum(dim=1).long()

            all_preds.append(predictions.cpu())
            all_targets.append(targets_class.cpu())

            validation_loss += loss.item()

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    n_batches = len(dataloader)

    return {
        'val_loss': validation_loss / n_batches,
        'val_acc': accuracy_score(all_targets, all_preds),
        'val_kappa': cohen_kappa_score(all_targets, all_preds, weights='quadratic'),
        'val_f1': f1_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_recall': recall_score(all_targets, all_preds, average='macro', zero_division=0),
        'val_precision': precision_score(all_targets, all_preds, average='macro', zero_division=0),
    }

## Training Loop

In [ ]:
best_kappa                = 0.0
best_epoch                = 0
epochs_without_improvement = 0

history = {
    'train_loss': [], 'val_loss': [], 'val_acc': [],
    'val_kappa': [], 'val_f1': [], 'val_recall': [], 'val_precision': []
}

print("\nStarting training — Ordinal + Focal + CLAHE-YUV — Optuna best params\n")
print(f"  gamma={BEST_GAMMA:.4f}, alpha={BEST_ALPHA:.4f}, beta={BEST_BETA:.4f}")
print(f"  lr={BEST_LR:.2e}, weight_decay={BEST_WEIGHT_DECAY:.2e}")
print("="*80)

with open(log_path, 'a') as f:
    f.write(f"Optuna best: gamma={BEST_GAMMA:.4f}, alpha={BEST_ALPHA:.4f}, beta={BEST_BETA:.4f}, "
            f"lr={BEST_LR:.2e}, weight_decay={BEST_WEIGHT_DECAY:.2e}\n")

for epoch in range(1, n_epochs + 1):
    print(f"\nEpoch {epoch}/{n_epochs}")
    print("-" * 80)

    train_loss = training_step(model, train_loader, optimizer, device, loss_function, scaler)
    metrics    = validation_step(model, valid_loader, device, loss_function)
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(np.mean(train_loss))
    history['val_loss'].append(metrics['val_loss'])
    history['val_acc'].append(metrics['val_acc'])
    history['val_kappa'].append(metrics['val_kappa'])
    history['val_f1'].append(metrics['val_f1'])
    history['val_recall'].append(metrics['val_recall'])
    history['val_precision'].append(metrics['val_precision'])

    print(f"\n  Train Loss:    {history['train_loss'][-1]:.5f}")
    print(f"  Val Loss:      {metrics['val_loss']:.5f}")
    print(f"  Val Accuracy:  {metrics['val_acc']*100:.2f}%")
    print(f"  Val Kappa:     {metrics['val_kappa']:.4f}")
    print(f"  Val F1:        {metrics['val_f1']:.4f}")
    print(f"  Learning Rate: {current_lr:.7f}")

    log_line = (
        f"epoch: {epoch} | lr: {current_lr:.7f} | "
        f"train_loss: {history['train_loss'][-1]:.5f} | "
        f"val_loss: {metrics['val_loss']:.5f} | "
        f"val_acc: {metrics['val_acc']:.4f} | "
        f"val_kappa: {metrics['val_kappa']:.4f}\n"
    )
    with open(log_path, 'a') as f:
        f.write(log_line)

    if metrics['val_kappa'] > best_kappa:
        best_kappa                 = metrics['val_kappa']
        best_epoch                 = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), model_path)
        print(f"\n  Best model saved! Kappa: {best_kappa:.4f}")
    else:
        epochs_without_improvement += 1
        print(f"\n  No improvement for {epochs_without_improvement} epoch(s)")

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping at epoch {epoch} — best epoch: {best_epoch}, kappa: {best_kappa:.4f}")
        break

print("\n" + "="*80)
print(f"Training complete. Best kappa: {best_kappa:.4f} at epoch {best_epoch}")
print("="*80)

## Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(history['train_loss'], label='Train Loss')
axes[0, 0].plot(history['val_loss'],   label='Val Loss')
axes[0, 0].set_title('Loss (Ordinal + Focal γ=0.2 + CLAHE-YUV)')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(history['val_acc'], label='Val Accuracy', color='green')
axes[0, 1].set_title('Validation Accuracy')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend(); axes[0, 1].grid(True)

axes[1, 0].plot(history['val_kappa'], label='Val Kappa', color='orange')
axes[1, 0].set_title('Validation Kappa')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('QW Kappa')
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], label='Val F1', color='red')
axes[1, 1].set_title('Validation F1 Score')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Macro F1')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-training.png', dpi=300, bbox_inches='tight')
plt.show()

## Evaluation on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

model.load_state_dict(torch.load(model_path, weights_only=True))
model.eval()

test_preds   = []
test_targets = []

with torch.no_grad():
    for batch_data, batch_targets, _ in tqdm(test_loader, desc="Testing"):
        batch_data = batch_data.to(device)
        logits     = model(batch_data)
        test_preds.append(decode_ordinal_predictions(logits).cpu())
        test_targets.append(batch_targets.sum(dim=1).long())

test_preds   = torch.cat(test_preds).numpy()
test_targets = torch.cat(test_targets).numpy()

# ── Point estimates ────────────────────────────────────────────────────────────
test_accuracy = accuracy_score(test_targets, test_preds)
test_kappa    = cohen_kappa_score(test_targets, test_preds, weights='quadratic')
test_f1       = f1_score(test_targets, test_preds, average='macro', zero_division=0)

# ── Bootstrap CI (1 000 resamples) ────────────────────────────────────────────
N_BOOTSTRAP = 1000
rng = np.random.default_rng(seed=42)
n   = len(test_targets)
boot_acc   = np.empty(N_BOOTSTRAP)
boot_kappa = np.empty(N_BOOTSTRAP)
boot_f1    = np.empty(N_BOOTSTRAP)

for i in tqdm(range(N_BOOTSTRAP), desc="Bootstrap"):
    idx           = rng.integers(0, n, size=n)
    boot_acc[i]   = accuracy_score(test_targets[idx], test_preds[idx])
    boot_kappa[i] = cohen_kappa_score(test_targets[idx], test_preds[idx], weights='quadratic')
    boot_f1[i]    = f1_score(test_targets[idx], test_preds[idx], average='macro', zero_division=0)

def boot_stats(arr):
    return arr.std(ddof=1), np.percentile(arr, 2.5), np.percentile(arr, 97.5)

acc_std,   acc_lo,   acc_hi   = boot_stats(boot_acc)
kappa_std, kappa_lo, kappa_hi = boot_stats(boot_kappa)
f1_std,    f1_lo,    f1_hi    = boot_stats(boot_f1)

print("\n" + "="*80)
print("TEST SET RESULTS — Ordinal + Focal (γ=0.2) + CLAHE-YUV")
print("="*80)
print(f"Accuracy : {test_accuracy*100:.2f}% ± {acc_std*100:.2f}%  [{acc_lo*100:.2f}% – {acc_hi*100:.2f}%]")
print(f"QW Kappa : {test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]")
print(f"Macro F1 : {test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]")
print("="*80)
print(classification_report(test_targets, test_preds,
      target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))

cm = confusion_matrix(test_targets, test_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'ISUP {i}' for i in range(6)],
            yticklabels=[f'ISUP {i}' for i in range(6)])
plt.title('Confusion Matrix — Ordinal + Focal (γ=0.2) + CLAHE-YUV', fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()

with open('logs/b0-entropy-ordinal-focal-yhuv-clahe-test-results.txt', 'w') as f:
    f.write("EfficientNet-B0 — Entropy + OrdinalLoss + FocalLoss(γ=0.2) + CLAHE-YUV\n")
    f.write("="*80 + "\n\n")
    f.write(f"Bootstrap resamples: {N_BOOTSTRAP}\n\n")
    f.write(f"Accuracy : {test_accuracy*100:.2f}% ± {acc_std*100:.2f}%  [{acc_lo*100:.2f}% – {acc_hi*100:.2f}%]\n")
    f.write(f"QW Kappa : {test_kappa:.4f} ± {kappa_std:.4f}  [{kappa_lo:.4f} – {kappa_hi:.4f}]\n")
    f.write(f"Macro F1 : {test_f1:.4f} ± {f1_std:.4f}  [{f1_lo:.4f} – {f1_hi:.4f}]\n\n")
    f.write(classification_report(test_targets, test_preds,
            target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))
    f.write("\nConfusion Matrix:\n" + str(cm))

print("Saved: logs/b0-entropy-ordinal-focal-yhuv-clahe-test-results.txt")

In [ ]:
cm_norm  = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
classes  = [f'ISUP {i}' for i in range(6)]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('Normalized Confusion Matrix — Ordinal + Focal (γ=0.2) + CLAHE-YUV')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('logs/b0-entropy-ordinal-focal-yhuv-clahe-confusion-matrix-normalized.png',
            dpi=300, bbox_inches='tight')
plt.show()

## Summary

| Component | Detail |
|---|---|
| Entropy filtering | Top 20% hardest samples removed |
| **OrdinalLoss** | Plain BCE on ordinal-encoded targets |
| **OrdinalFocalLoss** | Focal-weighted BCE, α=0.25, γ=0.2 |
| **Combined loss** | `total = OrdinalLoss + OrdinalFocalLoss` |
| **CLAHE-YUV preprocessing** | CLAHE (clip_limit=2.0, tile=8×8) on Y channel of YUV — colour (U/V) preserved |
| **Colour space pipeline** | RGB → YUV → CLAHE(Y) → YUV → RGB |